# Feynman-Kac in practice: when do grids beat Monte Carlo?

**About this project**

This is a COSC2500 project (Semester 2, 2026) by Connor Mcareavey. It prices options on stocks in two completely different ways, by solving a partial differential equation on a grid and by simulating random stock price paths, and asks a simple question: which method should we use, and when? The two methods are connected by the Feynman-Kac formula, which says they compute the same number. Because the options studied here have exact prices, every error reported in the project is a true error, measured against the exact answer. The project is split into this overview and four notebooks, which use code from the `pricing` folder.

**The Feynman-Kac connection**

In the Black-Scholes model a stock price follows geometric Brownian motion, $dS = rS\,dt + \sigma S\,dB_t$, under the risk neutral measure. A European option pays $f(S(T))$ at expiry $T$. Its value $V(S,t)$ can be described in two ways. As a PDE, $V$ solves the Black-Scholes equation

$$\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2S^2\frac{\partial^2V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV = 0, \qquad V(S,T) = f(S).$$

As an expectation, $V$ is the discounted average payoff over all possible paths,

$$V(S,t) = e^{-r(T-t)}\,\mathbb{E}\left[f(S(T)) \mid S(t) = S\right].$$

The Feynman-Kac theorem says these are the same function. To see why, suppose $V$ solves the PDE and look at $Y_t = e^{-rt}V(S(t),t)$. Itô's formula (which we take as given) gives

$$dY_t = e^{-rt}\left(\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2S^2\frac{\partial^2V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV\right)dt + e^{-rt}\sigma S\frac{\partial V}{\partial S}\,dB_t.$$

The bracket is exactly the left hand side of the PDE, so it is zero and only the $dB_t$ term is left. A stochastic integral like that has zero mean (under mild conditions on $V$), so $\mathbb{E}[Y_T] = Y_0$. Since $Y_T = e^{-rT}f(S(T))$ and $Y_0 = V(S(0),0)$, this is exactly the expectation formula.

**Why the two views lead to different numerical methods**

The PDE view leads to grid methods. We discretise $S$ and $t$, replace derivatives by finite differences (Module 1.3), and step backwards from expiry, solving a banded linear system at each step when the scheme is implicit (Module 2.3). The errors come from the grid spacing, and the main risks are instability and loss of accuracy where the payoff is not smooth (Modules 3 and 4.1). The expectation view leads to Monte Carlo. We simulate many paths of the stock, either exactly or with the Euler-Maruyama method (Module 4.9), average the payoffs, and estimate the error from the spread of the results (Module 4.10). The errors come from the finite number of paths and from time stepping.

The two methods scale very differently. A grid with $N$ points per variable needs $N^d$ points for an option on $d$ stocks (Module 4.2), while Monte Carlo error does not depend on $d$. Which one wins therefore depends on the dimension, and finding where the crossover happens is the main goal of the project.

**Background and previous work**

The Black-Scholes PDE was derived by Black and Scholes (1973) and Merton (1973) from a hedging argument: a portfolio of the option and the stock can be made riskless, so it must earn the risk free rate. The idea of writing the solution as an expectation is older. Feynman (1948) wrote solutions of the Schrödinger equation as averages over paths, and Kac (1949) made the corresponding result for the heat equation rigorous, which is where the name Feynman-Kac comes from. In finance the expectation form is the risk neutral pricing formula used throughout this project.

Finite difference methods were first applied to option pricing in the late 1970s, with Brennan and Schwartz (1978) giving an early systematic treatment. The Crank-Nicolson scheme (Crank and Nicolson 1947), originally developed for heat conduction problems, became the standard choice because it is second order and unconditionally stable. The change of variables to the heat equation used in Part 1 is the classical one presented by Wilmott, Howison and Dewynne (1995). The weakness of Crank-Nicolson with non-smooth data was identified by Rannacher (1984) for parabolic equations in general, and his remedy of a few implicit start-up steps was studied for option payoffs by Pooley, Vetzal and Forsyth (2003) and analysed by Giles and Carter (2006). Part 1 reproduces their main observations and adds a measurement of how many start-up steps the gamma of a digital option needs.

The Monte Carlo method itself goes back to Metropolis and Ulam (1949), and Boyle (1977) was the first to use it to price options. The standard modern reference is Glasserman (2004), which covers variance reduction methods such as the geometric average control variate of Kemna and Vorst (1990) that we use in Part 4. The Euler-Maruyama method is named after Maruyama (1955), and Higham (2001) gives an accessible account of its weak and strong orders of convergence. Duffie and Glynn (1995) showed how a fixed computing budget should be split between time steps and paths, which leads to the $\varepsilon^{-3}$ cost we measure in Part 2, and the multilevel Monte Carlo method of Giles (2008) was designed to bring that cost back down to roughly $\varepsilon^{-2}$.

The phrase "curse of dimensionality" is due to Bellman (1957). The comparison between a grid's error of order $n^{-2/d}$ and Monte Carlo's $n^{-1/2}$ is standard (Glasserman 2004, Section 1.1), and it is the usual reason given for pricing multi-asset products by Monte Carlo. Sparse grids (Bungartz and Griebel 2004) reduce the number of points a grid needs from about $n^d$ to about $n(\log n)^{d-1}$ for sufficiently smooth functions, which pushes the crossover to higher dimensions, although payoffs with kinks are exactly the functions that are not smooth enough to get the full benefit. What this project adds is a direct measurement of the crossover on a problem with an exact answer in every dimension, including the extra cost of time stepping in a PDE solver and the effect of the explicit scheme's stability limit.

**The test problem**

Throughout the project we use $S_0 = K = 100$, $r = 0.05$, $\sigma = 0.2$ and $T = 1$ year. The two single stock options are a European call, with payoff $\max(S - K, 0)$, and a cash-or-nothing digital call, which pays 1 if $S > K$ and nothing otherwise. The call payoff has a kink at the strike, and the digital payoff has a jump there. Both cause trouble for numerical methods, in the same way that discontinuities caused trouble for numerical integration in exercise R1.3.

In [1]:
import sys
sys.path.append("..")
from pricing import black_scholes as bs

S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.2, 1.0
print(f"European call price  {bs.call_price(S0, K, r, sigma, T):.6f}")
print(f"Digital call price   {bs.digital_price(S0, K, r, sigma, T):.6f}")
print(f"Call delta and gamma {bs.call_delta(S0, K, r, sigma, T):.6f}, {bs.call_gamma(S0, K, r, sigma, T):.6f}")

European call price  10.450584
Digital call price   0.532325
Call delta and gamma 0.636831, 0.018762


**How the project is organised**

Part 1 (`01_pde.ipynb`) transforms the Black-Scholes PDE into the heat equation of exercise R4.1(a), and compares the explicit, implicit and Crank-Nicolson schemes for stability, convergence and accuracy of the Greeks, including Rannacher smoothing near the strike. Part 2 (`02_sde.ipynb`) prices the same call by Monte Carlo and separates the statistical error from the Euler-Maruyama time stepping bias. Part 3 (`03_high_dimensions.ipynb`) derives how the cost of each method grows with the number of stocks, and measures the crossover on a basket problem with an exact answer in every dimension. Part 4 (`04_basket_extension.ipynb`) is an extension that prices a basket of up to 20 correlated stocks by Monte Carlo, using a control variate.

The code lives in the `pricing` folder. `black_scholes.py` has the closed form prices and Greeks, `pde.py` the finite difference solvers, `sde.py` the path simulation, `high_dim.py` the grid and Monte Carlo integration in $d$ dimensions, and `basket.py` the correlated basket simulation. The file `tests/test_pricing.py` checks every method against a known answer and runs in a few seconds with `pytest`. All random numbers come from seeded generators, so every result can be reproduced exactly.

**Verification and validation**

Babuska and Oden (2004), from the Module 0 reading, separate two questions that are easy to confuse. Verification asks whether the computer program solves the mathematical model accurately, which they summarise as solving the equations right. Validation asks whether the mathematical model describes the real system accurately enough, or solving the right equations.

This project puts most of its effort into verification, at three levels. First, every method is compared with an exact closed form price, and `tests/test_pricing.py` repeats these checks automatically. Second, we check convergence rates and not just single numbers. The observed orders (second order for Crank-Nicolson with Rannacher smoothing, first order for the implicit scheme, weak order one and strong order one half for Euler-Maruyama, and $M^{-1/2}$ for Monte Carlo) all match the theory. This is a much stronger test than matching one number, because most bugs change the order of convergence even when they leave the answer roughly right. Third, we check the error estimates themselves, by confirming that the Monte Carlo 95% intervals contain the true price about 95% of the time, and by testing the one outlying comparison in Part 4 with fresh simulations. For the arithmetic basket, which has no exact price, the geometric basket benchmark verifies the simulation, and the plain and control variate estimates agree to within about 1.2 standard errors in all 18 cases.

Validation is a different matter, and we have not done it. The Black-Scholes model assumes a constant volatility, normally distributed log returns, continuous trading and no transaction costs, and none of these hold exactly. In real option markets the volatility implied by observed prices depends on the strike (the volatility smile), and stock returns have heavier tails than the normal distribution allows. So the prices computed here are exact prices for the Black-Scholes model, not predictions of market prices. The conclusions about the numerical methods should still carry over to more realistic models, which mostly make them more pressing. A stochastic volatility model such as that of Heston (1993) adds a second state variable, so its PDE is two dimensional, and a volatility that depends on $S$ and $t$ removes the exact solution of the SDE, so the time stepping bias of Part 2 can no longer be avoided.

**Summary of findings**

For the PDE (Part 1), the heat equation transform reduces the problem to one with a constant coefficient. As a result the explicit stability limit $\lambda = \Delta\tau/\Delta x^2 \le \frac{1}{2}$ does not depend on how far the grid extends, whereas in the original variable $S$ the stable time step shrinks like $1/S_{\max}^2$. Just past the limit, at $\lambda = 0.52$, the explicit scheme's error grows to 43. With the large time steps that make Crank-Nicolson attractive, it fails near the strike. For the call it drops to first order, and its gamma has an error of 0.6 against a true value of 0.02. For the digital its price error stays at 0.071 however much the grid is refined. Rannacher smoothing, a few implicit steps at the start, restores second order convergence. We also found that the usual two start-up steps are not enough for the digital's gamma, where the error stalls at about 14% of the largest gamma, while four steps fix it.

For Monte Carlo (Part 2), the statistical error falls like $M^{-1/2}$ and the single run standard error is a reliable error bar, with 95% intervals covering the true price in 95.5% of 400 runs. The Euler-Maruyama bias falls at first order, with a fitted weak order of 1.1 and strong order 0.47. It could only be measured by driving the exact and approximate paths with the same random numbers, since otherwise the noise hides it. Together the two errors make the cost grow like $\varepsilon^{-3}$ instead of $\varepsilon^{-2}$, about 36 times more work for one cent accuracy.

For the dimension question (Part 3), the one dimensional PDE reaches an error of $3 \times 10^{-6}$ in under half a second, which would take Monte Carlo about 17 days. Per unit of work, a second order grid converges like $W^{-2/d}$ (quadrature), $W^{-2/(d+1)}$ (Crank-Nicolson PDE) or $W^{-2/(d+2)}$ (explicit PDE), against $W^{-1/2}$ for Monte Carlo, so the predicted crossovers are at 4, 3 and 2 dimensions. On a geometric basket with an exact price, the measured crossover for quadrature at practical accuracies is at about 5 dimensions, and a full PDE grid loses from about 3. The ball volume example showed that Monte Carlo's dimension free rate can still hide a constant that grows quickly with $d$.

In the extension (Part 4), a 20 stock basket is out of reach for any grid but takes Monte Carlo a fraction of a second. Using the geometric basket's closed form as a control variate reduced the variance by factors of 25 to over 2000, depending on the correlation, giving a 20 stock arithmetic basket price of $7.3567 \pm 0.0026$ from 200000 paths.

**What the results mean in practice**

Option prices are quoted in cents, so a natural accuracy target for our call, which is worth about 10.45, is $10^{-2}$, or about 0.1%. The Crank-Nicolson solver with Rannacher smoothing reaches this with a couple of hundred grid points in about a millisecond, while plain Monte Carlo needs around two million paths to bring its standard error down to one cent, which takes about a hundred times longer. The gap grows quickly as the target tightens. A bank does not price one option once: risk systems revalue whole portfolios under many market scenarios every day, so differences in speed of this size decide which methods are practical.

The Greeks matter as much as the price. Delta is the number of shares a trader holds to hedge each option, and gamma measures how quickly that hedge has to be adjusted as the stock moves. The plain Crank-Nicolson gamma for the call in Part 1, wrong by 0.6 against a true value of about 0.02, would make a hedger trade far too much near the strike. That is why the oscillation problem matters even when the price looks fine, and why practical PDE codes use Rannacher smoothing or something similar.

Finally, the crossover found in Part 3 matches how the work is usually divided in practice. Problems with one or two underlying variables are normally solved with PDE methods or trees, while baskets, and products that depend on the whole path of a stock, are normally priced by Monte Carlo. Our measurements give a quantitative reason for that split: with second order methods, grids win up to about three to five dimensions, depending on the method and the accuracy required, and lose beyond that.

**Limitations and possible extensions**

All the options here are European with constant volatility and interest rate, chosen so that exact answers exist. That makes the error measurements clean, but it is also the setting where numerical methods are needed least. The natural next steps are products without closed forms. American options would turn the PDE into a free boundary problem and would make Monte Carlo much harder. Local volatility models would break the heat equation transform and make Euler-Maruyama time stepping unavoidable. Grids could be improved with non-uniform spacing near the strike or with sparse grids in higher dimensions, and Monte Carlo with quasi-random points or further variance reduction.

**Self-assessment**

The project sheet invites an optional self-assessment. The grade claimed for this project is a 7, justified below against the four criteria used to award additional points.

For knowledge and content mastery, the project derives rather than quotes its main results: the transformation of the Black-Scholes PDE into the heat equation, the von Neumann stability analysis of the explicit and Crank-Nicolson schemes, the first order bias of Euler-Maruyama in the mean, the variance of the control variate estimator, and the convergence exponents that decide the crossover dimension. It uses material from every module of the course, including finite differences and discontinuities from Module 1, banded systems and the Cholesky factorisation from Module 2, time stepping and stiffness from Module 3, and PDEs, Monte Carlo methods, SDEs and error estimation from Module 4.

For critical evaluation and insight, several results came from refusing to take a first answer at face value. Measuring the error over a window of stock prices, rather than only at the strike, exposed a Crank-Nicolson failure that the usual single point check hides. Coupling the exact and approximate paths made an Euler-Maruyama bias measurable when it would otherwise have been lost in the noise. An outlying comparison in Part 4 was tested with fresh simulations rather than ignored or explained away. The limitations of each experiment are stated where they arise, including the pre-asymptotic grid fits in Part 3, the unusually kind test problem, and the absence of validation against market data.

For originality, the project goes beyond reproducing textbook results. It finds that the usual two Rannacher steps are not enough for the gamma of a digital option while four are, it shows that the explicit scheme's stability limit costs a whole dimension in the crossover, and it measures the crossover directly on a problem with an exact answer in every dimension. The extension prices a 20 stock basket with a variance reduction factor of over 2000.

For communication, every result is presented with a figure, printed numbers and a short explanation tied back to the course notes, the derivations are written out in full, and the code is short, commented and backed by automated tests, with fixed random seeds so that every number in the report can be reproduced from the public repository.

**How to run the code**

Install the packages with `pip install -r requirements.txt`, run the tests from the top folder with `pytest`, and then run the notebooks in order. Each notebook takes well under a minute on a laptop and saves its figures to the `figures` folder.

**References**

Babuska, I. and Oden, J. T. (2004). Verification and validation in computational engineering and science: basic concepts. *Computer Methods in Applied Mechanics and Engineering*, 193, 4057-4066.

Bellman, R. (1957). *Dynamic Programming*. Princeton University Press.

Black, F. and Scholes, M. (1973). The pricing of options and corporate liabilities. *Journal of Political Economy*, 81(3), 637-654.

Boyle, P. P. (1977). Options: a Monte Carlo approach. *Journal of Financial Economics*, 4(3), 323-338.

Brennan, M. J. and Schwartz, E. S. (1978). Finite difference methods and jump processes arising in the pricing of contingent claims: a synthesis. *Journal of Financial and Quantitative Analysis*, 13(3), 461-474.

Bungartz, H.-J. and Griebel, M. (2004). Sparse grids. *Acta Numerica*, 13, 147-269.

Crank, J. and Nicolson, P. (1947). A practical method for numerical evaluation of solutions of partial differential equations of the heat-conduction type. *Mathematical Proceedings of the Cambridge Philosophical Society*, 43(1), 50-67.

Duffie, D. and Glynn, P. (1995). Efficient Monte Carlo simulation of security prices. *The Annals of Applied Probability*, 5(4), 897-905.

Feynman, R. P. (1948). Space-time approach to non-relativistic quantum mechanics. *Reviews of Modern Physics*, 20(2), 367-387.

Giles, M. B. (2008). Multilevel Monte Carlo path simulation. *Operations Research*, 56(3), 607-617.

Giles, M. B. and Carter, R. (2006). Convergence analysis of Crank-Nicolson and Rannacher time-marching. *Journal of Computational Finance*, 9(4), 89-112.

Glasserman, P. (2004). *Monte Carlo Methods in Financial Engineering*. Springer, New York.

Heston, S. L. (1993). A closed-form solution for options with stochastic volatility with applications to bond and currency options. *The Review of Financial Studies*, 6(2), 327-343.

Higham, D. J. (2001). An algorithmic introduction to numerical simulation of stochastic differential equations. *SIAM Review*, 43(3), 525-546.

Kac, M. (1949). On distributions of certain Wiener functionals. *Transactions of the American Mathematical Society*, 65(1), 1-13.

Kemna, A. G. Z. and Vorst, A. C. F. (1990). A pricing method for options based on average asset values. *Journal of Banking and Finance*, 14(1), 113-129.

Maruyama, G. (1955). Continuous Markov processes and stochastic equations. *Rendiconti del Circolo Matematico di Palermo*, 4(1), 48-90.

Merton, R. C. (1973). Theory of rational option pricing. *The Bell Journal of Economics and Management Science*, 4(1), 141-183.

Metropolis, N. and Ulam, S. (1949). The Monte Carlo method. *Journal of the American Statistical Association*, 44(247), 335-341.

Nieminen, T. and Stilgoe, A. (2026). *COSC2500/COSC7500 Numerical Methods in Computational Science*, course notes. School of Mathematics and Physics, The University of Queensland.

Pooley, D. M., Vetzal, K. R. and Forsyth, P. A. (2003). Convergence remedies for non-smooth payoffs in option pricing. *Journal of Computational Finance*, 6(4), 25-40.

Rannacher, R. (1984). Finite element solution of diffusion problems with irregular data. *Numerische Mathematik*, 43(2), 309-327.

Sauer, T. (2018). *Numerical Analysis*, 3rd edition. Pearson.

Wilmott, P., Howison, S. and Dewynne, J. (1995). *The Mathematics of Financial Derivatives: A Student Introduction*. Cambridge University Press.